# 📊 Deep Research — Batch Pairwise Report Evaluation for the domain `Ecology`

This notebook evaluates **all reports in a directory** using pairwise rubrics.
Reports are grouped by their leading report number (e.g. `1_o3-mini_orkg_d1_b1.md` → report #1).
For each report number, all C(4,2) = 6 config pairs are evaluated.
Results are accumulated across all report numbers and saved to disk.

**Report filename format:** `{number}_{engine}_orkg_{config}.md`  
e.g. `1_o3-mini_orkg_d1_b1.md` → report #1, config `d1_b1`


## 1 — Imports & Setup

In [ ]:
!pip install yescieval==0.10.0 python-dotenv matplotlib 

In [ ]:
import re, json, csv, itertools
from pathlib import Path
from typing import Dict, Tuple, List
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown
from dotenv import load_dotenv
load_dotenv()
print("✅ Core imports loaded")


In [ ]:
# Pairwise rubric classes
from yescieval import CustomAutoJudge, VocabularyInjector, ExampleInjector
from yescieval.rubric.pairwise.depth   import TemporalPrecision, CausalReasoning, MechanisticUnderstanding
from yescieval.rubric.pairwise.breadth import ContextCoverage, ScopeCoverage, DimensionCoverage, ScaleCoverage, MethodCoverage
from yescieval.rubric.pairwise.rigor   import EpistemicCalibration, ExplicitUncertainty, QuantitativeEvidenceAndUncertainty
# from yescieval.rubric.pairwise.innovation import StateOfTheArtAndNovelty
# from yescieval.rubric.pairwise.gap        import GapIdentification
print("✅ YESciEval pairwise imports loaded")


## 2 — Configuration

In [ ]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────

REPORTS_DIR    = "your_reports_dir_here"          # directory containing all .md report files
QUESTIONS_CSV  = "your_questions_csv_path_here"   # path to 49-questions-ecology.csv
OUTPUT_DIR     = "your_output_dir_here"           # where to write CSVs, JSONs, plots
DOMAIN         = "ecology"
MODEL_ID       = "your_model_id_here"
DEVICE         = "cpu"                            # set to "cuda" if GPU is available
HF_TOKEN       = "your_hf_token_here"
MAX_NEW_TOKENS = 1024
REPORT_GLOB    = "*.md"
ALL_CONFIGS    = ['d1_b1', 'd1_b4', 'd4_b1', 'd4_b4']

RATING_MIN, RATING_MAX = 1, 5

# Category → list of (RubricClass, weight) tuples
CATEGORIES: Dict[str, List[Tuple[type, float]]] = {
    "depth": [
        (TemporalPrecision,         1/3),
        (CausalReasoning,           1/3),
        (MechanisticUnderstanding,  1/3),
    ],
    "breadth": [
        (ContextCoverage,   1/5),
        (ScopeCoverage,     1/5),
        (DimensionCoverage, 1/5),
        (MethodCoverage,    1/5),
        (ScaleCoverage,     1/5),
    ],
    "rigor": [
        (EpistemicCalibration,               1/3),
        (ExplicitUncertainty,                1/3),
        (QuantitativeEvidenceAndUncertainty, 1/3),
    ],
    # "innovation": [(StateOfTheArtAndNovelty, 1.0)],
    # "gap":        [(GapIdentification,       1.0)],
}

CAT_NAMES = list(CATEGORIES.keys())

print(f"Reports dir  : {REPORTS_DIR}")
print(f"Questions CSV: {QUESTIONS_CSV}")
print(f"Model        : {MODEL_ID}")
print(f"Categories   : {CAT_NAMES}")
print(f"All configs  : {ALL_CONFIGS}")


## 3 — Helper Functions

In [ ]:
def parse_config(stem: str) -> str:
    """Extracts dX_bY from filename, e.g. '1_o3-mini_orkg_d1_b1' -> 'd1_b1'."""
    m = re.search(r'd(\d+)_b(\d+)', stem)
    return f"d{m.group(1)}_b{m.group(2)}" if m else 'unknown'


def parse_report_number(stem: str) -> int:
    """Extracts the leading integer from a filename, e.g. '1_o3-mini' -> 1."""
    m = re.match(r'(\d+)[_\-]', stem)
    return int(m.group(1)) if m else -1


def parse_engine(stem: str) -> str:
    """Extracts engine string, e.g. '1_o3-mini_orkg_d1_b1' -> 'o3-mini'."""
    m = re.search(r'\d+_(.*?)_orkg', stem)
    return m.group(1) if m else stem


def load_question_from_csv(csv_path: str, report_number: int) -> str:
    """
    Reads the report title by ROW POSITION (1-based) from the Ecology CSV.
    The Ecology CSV uses a 'title' column.
    """
    for encoding in ['cp1252', 'utf-8', 'utf-8-sig', 'latin-1']:
        try:
            with open(csv_path, newline='', encoding=encoding) as f:
                reader = csv.DictReader(f)
                for row_num, row in enumerate(reader, start=1):
                    if row_num == report_number:
                        q = row.get('Your research question.', '').strip()
                        return q if q else f'Empty question at row {report_number}'
            return f'Row {report_number} not found in CSV'
        except UnicodeDecodeError:
            continue
    return 'Could not open CSV with any known encoding'


def parse_pairwise_judge_result(result, rubric_name: str) -> Tuple[int, int, str, str]:
    """Extracts (rating_a, rating_b, rationale_a, rationale_b) from a pairwise judge result."""
    if isinstance(result, str):
        cleaned = re.sub(r'<think>.*?</think>', '', result, flags=re.DOTALL).strip()
        json_str, depth, start = None, 0, None
        for i, ch in enumerate(cleaned):
            if ch == '{':
                if depth == 0: start = i
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0 and start is not None:
                    json_str = cleaned[start:i+1]; break
        if json_str:
            try: result = json.loads(json_str)
            except json.JSONDecodeError: pass
        if isinstance(result, str):
            return 0, 0, '', result

    if isinstance(result, dict):
        inner = result.get(rubric_name) or result.get(rubric_name.lower())
        if inner is None and len(result) == 1:
            inner = next(iter(result.values()))
        if isinstance(inner, dict):
            resp_a = inner.get('ResponseA', {})
            resp_b = inner.get('ResponseB', {})
            return (int(resp_a.get('rating', 0)), int(resp_b.get('rating', 0)),
                    str(resp_a.get('rationale', '')), str(resp_b.get('rationale', '')))
        if 'ResponseA' in result:
            resp_a = result['ResponseA']
            resp_b = result.get('ResponseB', {})
            return (int(resp_a.get('rating', 0)), int(resp_b.get('rating', 0)),
                    str(resp_a.get('rationale', '')), str(resp_b.get('rationale', '')))
        return 0, 0, '', str(result)

    if hasattr(result, 'ResponseA') and hasattr(result, 'ResponseB'):
        return (int(getattr(result.ResponseA, 'rating', 0)),
                int(getattr(result.ResponseB, 'rating', 0)),
                str(getattr(result.ResponseA, 'rationale', '')),
                str(getattr(result.ResponseB, 'rationale', '')))
    return 0, 0, '', str(result)


def compute_category_score(rubric_scores: List[Tuple[float, float]]) -> float:
    """
    Weighted mean of (score, weight) pairs, normalised to 0.0–1.0.
    Formula: (weighted_mean - 1) / (5 - 1)
    """
    total_w = sum(w for _, w in rubric_scores)
    if not total_w:
        return 0.0
    weighted_mean = sum(s * w for s, w in rubric_scores) / total_w
    return round((weighted_mean - 1) / (RATING_MAX - RATING_MIN), 4)


def run_pair(cfg_a, cfg_b, report_a_md, report_b_md, question, judge):
    """
    Runs the full pairwise evaluation for one (cfg_a, cfg_b) combination.
    Returns (rubric_raw, category_scores, overall_a, overall_b).
    """
    rubric_raw:      Dict[str, dict]             = {}
    category_scores: Dict[str, Dict[str, float]] = {}

    for cat_name, rubric_list in CATEGORIES.items():
        print(f'  📂 {cat_name.upper()}')
        cat_rubric_a: List[Tuple[float, float]] = []
        cat_rubric_b: List[Tuple[float, float]] = []

        for RubricClass, weight in rubric_list:
            rname = RubricClass.__name__
            print(f'     🔎 [{rname}] (w={weight:.3f}) ...')
            rubric = RubricClass(
                papers={}, question=question,
                answer_a=report_a_md, answer_b=report_b_md,
                domain=DOMAIN,
                vocabulary=VocabularyInjector(), example=ExampleInjector(),
            )
            raw = judge.judge(rubric=rubric, max_new_tokens=MAX_NEW_TOKENS)
            rating_a, rating_b, rat_a, rat_b = parse_pairwise_judge_result(raw, rname)
            rubric_raw[rname] = {
                'ResponseA': {'rating': rating_a, 'rationale': rat_a},
                'ResponseB': {'rating': rating_b, 'rationale': rat_b},
                'weight_in': {cat_name: weight},
            }
            print(f'        ResponseA={rating_a}/5  |  ResponseB={rating_b}/5')
            cat_rubric_a.append((float(rating_a), weight))
            cat_rubric_b.append((float(rating_b), weight))

        score_a = compute_category_score(cat_rubric_a)
        score_b = compute_category_score(cat_rubric_b)
        category_scores[cat_name] = {'a': score_a, 'b': score_b}
        print(f'     ✅ {cat_name}  {cfg_a}={score_a:.2f}  {cfg_b}={score_b:.2f}')

    overall_a = round(sum(v['a'] for v in category_scores.values()) / len(category_scores), 4)
    overall_b = round(sum(v['b'] for v in category_scores.values()) / len(category_scores), 4)
    return rubric_raw, category_scores, overall_a, overall_b


def discover_reports(reports_dir: str, glob_pattern: str) -> List[Path]:
    """Returns matching files sorted by their leading report number."""
    return sorted(
        Path(reports_dir).glob(glob_pattern),
        key=lambda p: parse_report_number(p.stem)
    )


print("✅ Helpers defined")


## 4 — Discover & Group Reports

Scans `REPORTS_DIR` for all `.md` files and groups them by report number.
Each group must contain all 4 configs (`d1_b1`, `d1_b4`, `d4_b1`, `d4_b4`).
Groups with missing configs are skipped with a warning.


In [ ]:
all_report_paths = discover_reports(REPORTS_DIR, REPORT_GLOB)

if not all_report_paths:
    raise FileNotFoundError(
        f"No files matching '{REPORT_GLOB}' found in: {REPORTS_DIR}\n"
        "Check REPORTS_DIR and REPORT_GLOB in the configuration cell."
    )

# Group paths by report number
report_groups: Dict[int, Dict[str, Path]] = defaultdict(dict)
for p in all_report_paths:
    rnum = parse_report_number(p.stem)
    cfg  = parse_config(p.stem)
    report_groups[rnum][cfg] = p

# Validate — skip groups missing configs
valid_groups: Dict[int, Dict[str, Path]] = {}
for rnum in sorted(report_groups):
    group = report_groups[rnum]
    missing = [c for c in ALL_CONFIGS if c not in group]
    if missing:
        print(f"⚠️  Report #{rnum}: missing configs {missing} — SKIPPING")
    else:
        valid_groups[rnum] = group
        engine = parse_engine(next(iter(group.values())).stem)
        print(f"✅ Report #{rnum}  engine={engine}  configs={list(group.keys())}")

print(f"\n{len(valid_groups)} valid report group(s) found, {len(ALL_CONFIGS)} configs each.")
print(f"Total pairs to evaluate: {len(valid_groups)} × {len(list(itertools.combinations(ALL_CONFIGS, 2)))} = "
      f"{len(valid_groups) * len(list(itertools.combinations(ALL_CONFIGS, 2)))}")


## 5 — Initialise the YESciEval Judge

In [ ]:
print(f"⏳ Loading {MODEL_ID} on {DEVICE} ...")
judge = CustomAutoJudge()
judge.from_pretrained(MODEL_ID, device=DEVICE, token=HF_TOKEN)
print("✅ Judge ready")


## 6 — Batch Pairwise Evaluation Loop

For each report group (one per report number), loads all 4 configs and runs
all C(4,2) = 6 pairwise combinations. Results are accumulated in
`all_batch_results` keyed by `(report_number, pair_key)`.


In [ ]:
# all_batch_results[report_num][pair_key] = { config_a, config_b,
#   rubric_scores, category_scores, overall_a, overall_b }
all_batch_results: Dict[int, Dict[str, dict]] = {}

for rnum, group in valid_groups.items():
    first_stem = next(iter(group.values())).stem
    engine_str = parse_engine(first_stem)
    question   = load_question_from_csv(QUESTIONS_CSV, rnum)

    print(f"\n{'#'*64}")
    print(f"Report #{rnum}  engine={engine_str}")
    print(f"Question: {question[:100]}{'...' if len(question) > 100 else ''}")
    print(f"{'#'*64}")

    # Load all 4 reports for this group
    loaded: Dict[str, str] = {}
    for cfg, path in group.items():
        loaded[cfg] = path.read_text(encoding='utf-8', errors='ignore')
        print(f"  ✅ {cfg}: {path.name}  ({len(loaded[cfg]):,} chars)")

    pairs = list(itertools.combinations(ALL_CONFIGS, 2))
    all_batch_results[rnum] = {}

    for pair_idx, (cfg_a, cfg_b) in enumerate(pairs, 1):
        pair_key = f'{cfg_a}_vs_{cfg_b}'
        print(f"\n  {'='*56}")
        print(f"  🔁 [{pair_idx}/{len(pairs)}] {cfg_a}  vs  {cfg_b}")
        print(f"  {'='*56}")

        rubric_raw, category_scores, overall_a, overall_b = run_pair(
            cfg_a, cfg_b, loaded[cfg_a], loaded[cfg_b], question, judge
        )

        all_batch_results[rnum][pair_key] = {
            'report_number':   rnum,
            'engine':          engine_str,
            'question':        question,
            'config_a':        cfg_a,
            'config_b':        cfg_b,
            'rubric_scores':   rubric_raw,
            'category_scores': category_scores,
            'overall_a':       overall_a,
            'overall_b':       overall_b,
        }
        print(f"  🏁 {pair_key}  {cfg_a}={overall_a:.2f}  {cfg_b}={overall_b:.2f}")

    # Flush GPU memory after each report group
    try:
        import torch, gc
        torch.cuda.empty_cache(); gc.collect()
    except Exception:
        pass

total_pairs = sum(len(v) for v in all_batch_results.values())
print(f"\n✅ Batch complete — {len(all_batch_results)} report group(s), {total_pairs} pairs evaluated.")


## 7 — Per-Category Pairwise Score Plots

For each quality category, one chart per report group with the **6 pair combinations**
on the x-axis and **normalised score (0–1)** on the y-axis.
Two bars per pair: **dark blue = Config A**, **light blue = Config B**.


In [ ]:
# ── §7: Per-Category Pairwise Score Plots (one figure per report group) ──────

RAW_DIMS = ['depth', 'breadth', 'rigor']
# RAW_DIMS = ['depth', 'breadth', 'rigor', 'innovation', 'gap']

RAW_PANEL_TITLES = {
    'depth':   'Research Depth Score',
    'breadth': 'Research Breadth Score',
    'rigor':   'Scientific Rigor Score',
}

BAR_COLOR_A = '#2980b9'
BAR_COLOR_B = '#85c1e9'
BAR_W_RAW   = 0.35

pairs_order = list(itertools.combinations(ALL_CONFIGS, 2))
n_pairs     = len(pairs_order)
xs_pairs    = np.arange(n_pairs)
n_dims      = len(RAW_DIMS)
n_cols_raw  = min(3, n_dims)
n_rows_raw  = int(np.ceil(n_dims / n_cols_raw))

per_cat_figs = {}   # store for saving in §9

for rnum, pair_results in all_batch_results.items():
    engine_str = next(iter(pair_results.values()))['engine']

    dim_pair_data = {dim: [] for dim in RAW_DIMS}
    for cfg_a, cfg_b in pairs_order:
        pair_key   = f'{cfg_a}_vs_{cfg_b}'
        cat_scores = pair_results[pair_key]['category_scores']
        for dim in RAW_DIMS:
            scores = cat_scores.get(dim, {'a': 0.0, 'b': 0.0})
            dim_pair_data[dim].append((scores['a'], scores['b']))

    plt.style.use('seaborn-v0_8-whitegrid')
    fig_raw, axes_raw = plt.subplots(n_rows_raw, n_cols_raw,
                                      figsize=(6 * n_cols_raw, 5 * n_rows_raw),
                                      squeeze=False)
    axes_raw_flat = axes_raw.ravel()

    for pi, dim in enumerate(RAW_DIMS):
        ax        = axes_raw_flat[pi]
        pair_data = dim_pair_data[dim]
        scores_a  = [d[0] for d in pair_data]
        scores_b  = [d[1] for d in pair_data]

        ax.bar(xs_pairs - BAR_W_RAW / 2, scores_a, width=BAR_W_RAW,
               color=BAR_COLOR_A, label='Config A (left in pair)')
        ax.bar(xs_pairs + BAR_W_RAW / 2, scores_b, width=BAR_W_RAW,
               color=BAR_COLOR_B, label='Config B (right in pair)')

        for x, va, vb in zip(xs_pairs, scores_a, scores_b):
            if va: ax.text(x - BAR_W_RAW / 2, va + 0.02, f'{va:.2f}',
                           ha='center', va='bottom', fontsize=8)
            if vb: ax.text(x + BAR_W_RAW / 2, vb + 0.02, f'{vb:.2f}',
                           ha='center', va='bottom', fontsize=8)

        ax.set_xticks(list(xs_pairs))
        ax.set_xticklabels([f'({a},\n{b})' for a, b in pairs_order], fontsize=8)
        ax.set_ylim(0, 1.15)
        ax.set_yticks([0.0, 0.25, 0.5, 0.75, 1.0])
        ax.set_ylabel('Score (0–1)', fontsize=9)
        ax.set_title(RAW_PANEL_TITLES[dim], fontsize=12)
        ax.legend(fontsize=8, loc='upper right')

    for j in range(pi + 1, len(axes_raw_flat)):
        axes_raw_flat[j].set_visible(False)

    fig_raw.suptitle(
        f'Per-Category Pairwise Scores (0–1) — {DOMAIN} / {engine_str} / Report #{rnum}\n'
        f'x-axis: 6 pair combinations  |  dark blue = Config A, light blue = Config B',
        fontsize=13,
    )
    fig_raw.tight_layout(rect=[0, 0.02, 1, 0.94])
    plt.show()
    per_cat_figs[rnum] = fig_raw
    print(f'📊 Report #{rnum} per-category plot rendered')


## 8 — Aggregated Per-Config Scores (Pointwise-Style Plot)

Scores from all pairs and all report groups are aggregated per config.
Each config appears in 3 pairs × N report groups; mean ± std are plotted.


In [ ]:
# ── §8: Aggregated Per-Config Scores across all report groups ────────────────

AGG_DIMS = ['depth', 'breadth', 'rigor']
# AGG_DIMS = ['depth', 'breadth', 'rigor', 'innovation', 'gap']

PANEL_TITLES_AGG = {
    'depth':   'Research Depth Score',
    'breadth': 'Research Breadth Score',
    'rigor':   'Scientific Rigor Score',
    'overall': 'Overall Quality Score',
}

# Collect per-config scores from every pair in every report group
config_dim_scores: dict = defaultdict(lambda: defaultdict(list))

for rnum, pair_results in all_batch_results.items():
    for pair_key, res in pair_results.items():
        cfg_a      = res['config_a']
        cfg_b      = res['config_b']
        cat_scores = res['category_scores']

        for dim in AGG_DIMS:
            if dim in cat_scores:
                config_dim_scores[cfg_a][dim].append(cat_scores[dim]['a'])
                config_dim_scores[cfg_b][dim].append(cat_scores[dim]['b'])

        config_dim_scores[cfg_a]['overall'].append(res['overall_a'])
        config_dim_scores[cfg_b]['overall'].append(res['overall_b'])

# Compute mean ± std
agg: dict = {}
for cfg in ALL_CONFIGS:
    agg[cfg] = {}
    for dim in PANEL_TITLES_AGG:
        vals = config_dim_scores[cfg].get(dim, [])
        agg[cfg][dim] = (float(np.mean(vals)), float(np.std(vals))) if vals else (0.0, 0.0)

print('Aggregated scores (mean ± std across all pairs and report groups):')
for cfg in ALL_CONFIGS:
    print(f'  {cfg}:')
    for dim, (mu, sd) in agg[cfg].items():
        print(f'    {dim:12s}  {mu:.3f} ± {sd:.3f}')

# Plot
n_panels  = len(PANEL_TITLES_AGG)
n_cols    = 3
n_rows    = int(np.ceil(n_panels / n_cols))
xs_agg    = np.arange(len(ALL_CONFIGS))
BAR_COLOR = '#2980b9'
ERR_COLOR = 'black'

plt.style.use('seaborn-v0_8-whitegrid')
fig_agg, axes_agg = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows), squeeze=False)
axes_agg_flat = axes_agg.ravel()

for i, (dim, panel_title) in enumerate(PANEL_TITLES_AGG.items()):
    ax    = axes_agg_flat[i]
    means = [agg[cfg][dim][0] for cfg in ALL_CONFIGS]
    stds  = [agg[cfg][dim][1] for cfg in ALL_CONFIGS]

    ax.bar(xs_agg, means, width=0.5, color=BAR_COLOR,
           yerr=stds, capsize=5,
           error_kw=dict(ecolor=ERR_COLOR, lw=1.5, capthick=1.5))

    for x, mu, sd in zip(xs_agg, means, stds):
        if mu:
            ax.text(x, mu + sd + 0.03, f'{mu:.2f}',
                    ha='center', va='bottom', fontsize=9)

    ax.set_title(panel_title, fontsize=12)
    ax.set_xticks(list(xs_agg))
    ax.set_xticklabels(ALL_CONFIGS, fontsize=10)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score (0–1)', fontsize=9)

for j in range(i + 1, len(axes_agg_flat)):
    axes_agg_flat[j].set_visible(False)

n_groups = len(all_batch_results)
fig_agg.suptitle(
    f'Research Quality Dimensions — {DOMAIN} (pairwise aggregated, n={n_groups} report groups)',
    fontsize=14,
)
fig_agg.tight_layout(rect=[0, 0.02, 1, 0.96])
plt.show()
print('📊 Aggregated plot rendered')


## 9 — Save Outputs to Disk

| File | Contents |
|---|---|
| `batch_pairwise_all_scores.csv` | One row per (report_group, pair) |
| `batch_pairwise_config_summary.csv` | Mean ± std per config across all pairs and report groups |
| `{rnum}_{pair_key}_scores.json` | Full rubric detail per pair per report group |
| `batch_pairwise_per_category_{rnum}.png` | Per-category plot per report group |
| `batch_pairwise_aggregated_quality_dimensions.png` | Aggregated config plot |


In [ ]:
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

# ── batch_pairwise_all_scores.csv ────────────────────────────────────────────
csv_rows = []
for rnum, pair_results in all_batch_results.items():
    for pair_key, res in pair_results.items():
        row = {
            'report_number': rnum,
            'engine':        res['engine'],
            'pair':          pair_key,
            'config_a':      res['config_a'],
            'config_b':      res['config_b'],
            'question':      res['question'],
        }
        for cat, scores in res['category_scores'].items():
            row[f'{cat}_score_a'] = round(scores['a'], 4)
            row[f'{cat}_score_b'] = round(scores['b'], 4)
        row['overall_score_a'] = round(res['overall_a'], 4)
        row['overall_score_b'] = round(res['overall_b'], 4)
        csv_rows.append(row)

all_scores_csv = out_dir / 'batch_pairwise_all_scores.csv'
with all_scores_csv.open('w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(csv_rows[0].keys()))
    w.writeheader(); w.writerows(csv_rows)
print(f'💾 CSV (all scores)     → {all_scores_csv}')

# ── batch_pairwise_config_summary.csv ────────────────────────────────────────
summary_rows = []
for cfg in ALL_CONFIGS:
    row = {'config': cfg}
    for dim in list(PANEL_TITLES_AGG.keys()):
        mu, sd = agg[cfg][dim]
        row[f'{dim}_mean'] = round(mu, 4)
        row[f'{dim}_std']  = round(sd, 4)
    summary_rows.append(row)

summary_csv = out_dir / 'batch_pairwise_config_summary.csv'
with summary_csv.open('w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    w.writeheader(); w.writerows(summary_rows)
print(f'💾 CSV (config summary) → {summary_csv}')

# ── Per-report-group per-category plot PNGs ───────────────────────────────────
for rnum, fig in per_cat_figs.items():
    fig_path = out_dir / f'batch_pairwise_per_category_{rnum}.png'
    fig.savefig(fig_path, dpi=200, bbox_inches='tight')
    print(f'💾 Per-category plot    → {fig_path}')

# ── Aggregated plot PNG ───────────────────────────────────────────────────────
agg_fig_path = out_dir / 'batch_pairwise_aggregated_quality_dimensions.png'
fig_agg.savefig(agg_fig_path, dpi=200, bbox_inches='tight')
print(f'💾 Aggregated plot      → {agg_fig_path}')

# ── Per-pair JSON files ───────────────────────────────────────────────────────
for rnum, pair_results in all_batch_results.items():
    for pair_key, res in pair_results.items():
        json_path = out_dir / f'{rnum}_{pair_key}_scores.json'
        json_path.write_text(
            json.dumps({'domain': DOMAIN, 'model': MODEL_ID, **res},
                       indent=2, ensure_ascii=False),
            encoding='utf-8'
        )
        print(f'💾 JSON                 → {json_path.name}')

print('\n✅ All outputs saved.')
